In [3]:
class User:                  #1.1
    def __init__(self,id,name,email):
        self._id=int(id)
        self._name=name.strip().title()
        if '@' in email:
            self._email=email.lower()
        else:
            raise ValueError('Ошибка')
    def __str__(self):
        return f"User(id={self._id}, name='{self._name}', email='{self._email}')"
    def __del__(self):
        print(f"User {self._name} deleted")
    @classmethod                    #1.2
    def from_string(cls, data: str):
        parts=[x.strip() for x in data.split(',')]
        return cls(*parts)
u = User.from_string("2, Alice Wonderland , alice@wonder.com")
print(u)

User Alice Wonderland deleted
User(id=2, name='Alice Wonderland', email='alice@wonder.com')


In [4]:
class Product:           #1.3
    def __init__(self,id:int,name,price:float,category):
        self.id=id
        self.name=name
        self.price=price
        self.category=category
    def __str__(self):
        return f"Product(id={self.id}, name='{self.name}', price={self.price}, category='{self.category}')"
    def __eq__(self, other):
        if not isinstance(other,Product):
            return False
        return self.id==other.id
    def __hash__(self):
        return hash(self.id)
    def to_dict(self):
        return {'id':self.id,'name':self.name,'price':self.price,'category':self.category}
p=Product(1,'Laptop',1200,'Electronics')
p1=Product(2,'Laptop',1200,'Electronics')
p.to_dict()

{'id': 1, 'name': 'Laptop', 'price': 1200, 'category': 'Electronics'}

In [5]:
class Inventory:       #1.4-1.5
    def __init__(self):
        self.items=[]
    def add_product(self,product: Product):
        if len(self.items)==0:
            self.items.append(product)
        else:
            a=True
            for x in self.items:
                if x.id==product.id:
                    a=False
                    break
            if a:
                self.items.append(product)

    def remove_product(self,product_id: int):
        for x in self.items:
            if x.id==product_id:
                self.items.remove(x)
                break
    def get_product(self,product_id: int):
        for x in self.items:
            if x.id==product_id:
                return x

    def get_all_products(self):
        return self.items
    def unique_products(self):
        return set(self.items)
    def to_dict(self):
        res={}
        for x in self.items:
            res[x.id]=x
        return res
    def filter_by_price(self,min_price: float):
        res=[y for y in filter(lambda x:x.price>=min_price,self.items)]
        return res
bag=Inventory()
bag.add_product(p)
for x in bag.get_all_products():
    print(x)


Product(id=1, name='Laptop', price=1200, category='Electronics')


In [6]:
from datetime import datetime       #1.6
class Logger:
    def log_action(self,user, action, product,filename:str):
        line=f"{datetime.now()};{user._id};{action};{product.id}\n"
        with open(filename,'a',encoding='UTF-8') as f:
            f.write(line)
    def read_logs(self,filename: str):
        res=[]
        with open(filename,'r',encoding='UTF-8') as f:
            for line in f:
                parts=line.strip().split(';')
                entry={
                    'timestamp':parts[0],
                    'user_id': int(parts[1]),
                    'action': parts[2],
                    'product_id': parts[3]
                }
                res.append(entry)
        return res


log=Logger()
log.log_action(u,'buy',p,'report1.txt')
log.read_logs('report1.txt')

[{'timestamp': '2026-04-03 15:04:53.871523',
  'user_id': 2,
  'action': 'buy',
  'product_id': '1'},
 {'timestamp': '2026-04-03 15:05:02.441830',
  'user_id': 2,
  'action': 'buy',
  'product_id': '1'},
 {'timestamp': '2026-04-03 15:07:47.183534',
  'user_id': 2,
  'action': 'buy',
  'product_id': '1'},
 {'timestamp': '2026-04-15 18:16:50.121410',
  'user_id': 2,
  'action': 'buy',
  'product_id': '1'},
 {'timestamp': '2026-04-15 19:31:37.588810',
  'user_id': 2,
  'action': 'buy',
  'product_id': '1'},
 {'timestamp': '2026-04-15 19:47:14.978573',
  'user_id': 2,
  'action': 'buy',
  'product_id': '1'},
 {'timestamp': '2026-04-15 19:49:23.492426',
  'user_id': 2,
  'action': 'buy',
  'product_id': '1'},
 {'timestamp': '2026-04-15 19:52:53.623856',
  'user_id': 2,
  'action': 'buy',
  'product_id': '1'},
 {'timestamp': '2026-04-15 19:53:28.794846',
  'user_id': 2,
  'action': 'buy',
  'product_id': '1'},
 {'timestamp': '2026-04-15 19:53:41.189582',
  'user_id': 2,
  'action': 'buy',
  

In [7]:
class Order:   #1.7-1.8-1.9
    def __init__(self, id: int,user:User,products):
        self.id=id
        self.user=user
        self.products=products
    def add_product(self,product: Product):
        self.products.append(product)
    def remove_product(self,product_id: int):
        for x in self.products:
            if x.id==product_id:
                self.products.remove(x)
                break
    def total_price(self):
        return sum([x.price for x in self.products])

    def __str__(self):
        products_str=", ".join([str(p) for p in self.products])
        return f"Заказ №{self.id} (Пользователь:{self.user._name}). Список продуктов: [{products_str}]"
    def most_expensive_products(self,n: int):
        return sorted(self.products,key=lambda x:x.price,reverse=True)[:n]

    def price_stream(self):
        for x in self.products:
            yield x.price

order=Order(1,u,[])
order.add_product(p)
#print(f"Итоговая цена: {order.total_price()}")
print(order)
'''for x in order.most_expensive_products(2):
    print(x)
for x in order.price_stream():
    print(x)'''

Заказ №1 (Пользователь:Alice Wonderland). Список продуктов: [Product(id=1, name='Laptop', price=1200, category='Electronics')]


'for x in order.most_expensive_products(2):\n    print(x)\nfor x in order.price_stream():\n    print(x)'

In [8]:
class OrderIterator:  #1.10
    def __init__(self,orders:Order):
        self.orders=orders
        self.index=0

    def __iter__(self):
        return self

    def __next__(self):
        if self.index<len(self.orders):
            ord=self.orders[self.index]
            self.index+=1
            return ord
        else:
            raise StopIteration
orders=[order,order]
res=OrderIterator(orders)
for x in res:
    print(x)


Заказ №1 (Пользователь:Alice Wonderland). Список продуктов: [Product(id=1, name='Laptop', price=1200, category='Electronics')]
Заказ №1 (Пользователь:Alice Wonderland). Список продуктов: [Product(id=1, name='Laptop', price=1200, category='Electronics')]


In [9]:
import numpy as np     #1.11-1.12-1.13
products = [Product(1,"Laptop",1200.0,"Electronics"), Product(2,"Mouse",25.0,"Electronics")]
prices=np.array([x.price for x in products])
mean_price=np.mean(prices)
median_price=np.median(prices)
print(prices)
print(mean_price,median_price)
prices_min=prices.min()
prices_max=prices.max()
normalized_prices=(prices-prices_min)/(prices_max-prices_min)
print(normalized_prices)

[1200.   25.]
612.5 612.5
[1. 0.]


In [10]:
products = [Product(1,"Laptop",1200.0,"Electronics"), Product(2,"T-Shirt",20.0,"Clothing")]      #1.14-1.15
a=np.array([x.category for x in products])
b=np.array(["Electronics", "Clothing", "Electronics"])
print(np.unique(b).size)

2


In [11]:
products = [Product(1,"Laptop",1200.0,"Electronics"), Product(2,"Mouse",25.0,"Electronics"), Product(3,"Monitor",450.0,"Electronics")]     #1.16
a=np.array([x.price for x in products])
mask=a>np.mean(a)
result=[p for p,m in zip(products,mask) if m]
print(result)

In [12]:
a=np.array([1200.0, 25.0, 450.0])    #1.17
print(a*0.9)

[1080.    22.5  405. ]


In [13]:
orders = [Order(1,u,[Product(1,"Laptop",1200.0,"Electronics")])]     #1.18
totals=[sum(p.price for p in order.products)]
print(totals)

[1200]


In [14]:
a=np.array([1200.0, 1225.0])   #1.19
print(np.mean(a))

1212.5


In [16]:
a=np.array([1200.0, 900.0, 1500.0])     #1.20
print(np.where(a>1000))

(array([0, 2]),)


In [17]:
import pandas as pd                 #1.21
from datetime import datetime
users = [User(1,"John Doe","john@example.com"),
         User(2,"Alice","alice@example.com")]
data={"id":[],
      "name":[],
      "email":[],
      "registration_date":[]}
for x in users:
    data['id'].append(x._id)
    data['name'].append(x._name)
    data["email"].append(x._email)
    data["registration_date"].append(datetime.now())
a=pd.DataFrame(data)
print(a)

   id      name              email          registration_date
0   1  John Doe   john@example.com 2026-04-15 23:00:14.698866
1   2     Alice  alice@example.com 2026-04-15 23:00:14.698870


In [18]:
products = [Product(1,"Laptop",1200.0,"Electronics"),       #1.22
            Product(2,"T-Shirt",20.0,"Clothing")]
data={"id":[],
      "name":[],
      "category":[],
      "price":[]}
for x in products:
    data['id'].append(x.id)
    data['name'].append(x.name)
    data["category"].append(x.category)
    data["price"].append(x.price)
b=pd.DataFrame(data)
print(b)

   id     name     category   price
0   1   Laptop  Electronics  1200.0
1   2  T-Shirt     Clothing    20.0


In [19]:
users_df = pd.DataFrame({    #1.23-1.24
    "id": [1, 2],
    "name": ["John", "Alice"]
})
orders_df=pd.DataFrame({
    "order_id":[101,102],
    "user_id":[1,2],
    "total":[1200,25]
})
merged_df=pd.merge(orders_df,users_df,left_on="user_id",right_on="id")
merged_df=merged_df[["order_id", "name", "total"]].rename(columns=({"name": "user_name"}))
print(merged_df)
srt=merged_df[merged_df['total']>100]
print(srt)

   order_id user_name  total
0       101      John   1200
1       102     Alice     25
   order_id user_name  total
0       101      John   1200


In [20]:
user_df=pd.DataFrame({  #1.25-1.26-1.27
    "order_id":[101,103,102],
    "user_name":["John","John","Alice"],
    "total":[1200,500,25]
})
print(user_df)
cnt=user_df.groupby("user_name")["total"].count().reset_index()
cnt=cnt.rename(columns={"total":"orders_count"})
df=user_df.groupby("user_name")["total"].mean().reset_index()
df=df.rename(columns={"total":"mean"})
res=user_df.groupby("user_name")["total"].sum().reset_index()
res=res.rename(columns={"total":"total_sum"})
print(res)
print(df)
print(cnt)

   order_id user_name  total
0       101      John   1200
1       103      John    500
2       102     Alice     25
  user_name  total_sum
0     Alice         25
1      John       1700
  user_name   mean
0     Alice   25.0
1      John  850.0
  user_name  orders_count
0     Alice             1
1      John             2


In [21]:
data=pd.DataFrame({                    #1.28
    "id" : [1,2,3],
    "name" : ["Laptop","Mouse","Shirt"],
    "category" : ["Electronics","Electronics","Clothing"],
    "price" : [1200,25,20]
})
df=data.groupby("category")["price"].mean().reset_index()
df=df.rename(columns={"price":"mean_price"})
print(df)

      category  mean_price
0     Clothing        20.0
1  Electronics       612.5


In [22]:
data=pd.DataFrame({     #1.29
    "id":[1,2],
    "name":["Laptop","Mouse"],
    "price":[1200,25]
})
data["discounted_price"]=data["price"]*0.9
print(data)

   id    name  price  discounted_price
0   1  Laptop   1200            1080.0
1   2   Mouse     25              22.5


In [23]:
data=pd.DataFrame({      #1.30
    "id":[1,2,3],
    "name":["Laptop","Mouse","Monitor"],
    "price":[1200,25,450]
})
data=data.sort_values("price",ascending=False)
print(data)

   id     name  price
0   1   Laptop   1200
2   3  Monitor    450
1   2    Mouse     25


In [24]:
data=pd.DataFrame({  #1.31
    "order_id":[101,102],
    "product_name":["Laptop","Mouse"],
    "price":[1200,25]
})
data["quantity"]=[1,1]
print(data)

   order_id product_name  price  quantity
0       101       Laptop   1200         1
1       102        Mouse     25         1


In [25]:
data=pd.DataFrame({  #1.32
    "order_id":[101,102],
    "product_name":["Laptop","Mouse"],
    "price":[1200,25],
    "quantity":[1,2]
})
data["total_price"]=data["price"]*data["quantity"]
print(data)

   order_id product_name  price  quantity  total_price
0       101       Laptop   1200         1         1200
1       102        Mouse     25         2           50


In [26]:
data=pd.DataFrame({       #1.33
    "product_name":["Laptop","T-Shirt"],
    "category":["Electronics","Clothing"],
    "price":[1200,20]
})
electronics_products=data[data['category']=="Electronics"]
print(electronics_products)

  product_name     category  price
0       Laptop  Electronics   1200


In [27]:
data=pd.DataFrame({        #1.34
    "product_name":["Laptop","Mouse","Shirt"],
    "category":["Electronics","Electronics","Clothing"]
})

new=data.groupby("category")["category"].value_counts().reset_index()
print(new)

      category  count
0     Clothing      1
1  Electronics      2


In [28]:
data=pd.DataFrame({     #1.35
    "product_name":["Laptop","Mouse","Shirt"],
    "category":["Electronics","Electronics","Clothing"],
    "price":[1200,25,20]
})
data=data.groupby("category")["price"].mean().reset_index()
data=data.rename(columns={"price":"mean_price"})
print(data)

      category  mean_price
0     Clothing        20.0
1  Electronics       612.5


In [29]:
data=pd.DataFrame({       #1.36
    "order_id":[101,102],
    "total_price":[1200,50]
})
data=data.sort_values("total_price",ascending=False)
print(data)

   order_id  total_price
0       101         1200
1       102           50


In [30]:
data=pd.DataFrame({      #1.37
    "order_id":[101,102,103,104],
    "total_price":[1200,50,500,1500]
})
data=data.sort_values("total_price",ascending=False)
print(data.head(3))

   order_id  total_price
3       104         1500
0       101         1200
2       103          500


In [31]:
users_df=pd.DataFrame({          #1.38
    "user_id":[1,2],
    "user_name":["John","Alice"]
})

orders_df=pd.DataFrame({
    "order_id":[101,102],
    "user_id":[1,2],
    "total_price":[1200,50]
})

merged_df=pd.merge(users_df,orders_df)
merged_df=merged_df[["order_id","user_name","total_price"]]
print(merged_df)

   order_id user_name  total_price
0       101      John         1200
1       102     Alice           50


In [32]:
data=pd.DataFrame({             #1.39
    "user_name":["John","John","Alice"],
    "total_price":[1200,500,50]
})
data=data.groupby("user_name")["total_price"].mean().reset_index()
data=data.rename(columns={"total_price":"mean_total"})
print(data)


  user_name  mean_total
0     Alice        50.0
1      John       850.0


In [33]:
data=pd.DataFrame({            #1.40
    "user_name":["John","John","Alice"],
    "total_price":[101,103,102]
})
data=data.groupby("user_name")["user_name"].value_counts().reset_index()
print(data)

  user_name  count
0     Alice      1
1      John      2


In [34]:
data=pd.DataFrame({            #1.41
    "user_name":["John","John","Alice"],
    "total_price":[1200,500,50]
})
data=data.groupby("user_name")["total_price"].max().reset_index()
data=data.rename(columns={"total_price":"max_order"})
print(data)

  user_name  max_order
0     Alice         50
1      John       1200


In [35]:
data=pd.DataFrame({              #1.42
    "user_name":["John","John","John","Alice"],
    "category":["Electronics","Electronics","Clothing","Clothing"]
})
data=data.groupby("user_name")["category"].nunique().reset_index()
data=data.rename(columns={"category":"unique_categories"})
print(data)

  user_name  unique_categories
0     Alice                  1
1      John                  2


In [36]:
data=pd.DataFrame({            #1.43
    "user_name":["John","Alice"],
    "total_sum":[1700,25]
})
data["VIP"]=data["total_sum"]>1000
print(data)

  user_name  total_sum    VIP
0      John       1700   True
1     Alice         25  False


In [37]:
data=pd.DataFrame({            #1.44
    "user_name":["John","Alice","Bob"],
    "total_sum":[1700,25,1700],
    "mean_total":[850,25,600]
})
data=data.sort_values(["total_sum","mean_total"],ascending=[False,True])
print(data)

  user_name  total_sum  mean_total
2       Bob       1700         600
0      John       1700         850
1     Alice         25          25


In [38]:
data=pd.DataFrame({  #1.45
    "user_name":["John","John","Alice"],
    "order_id":[101,103,102],
    "total_price":[1200,500,25],
    "category":["Electronics","Clothing","Clothing"]
})
final=data.groupby("user_name").agg(
    total_orders=("order_id", "count"),
    total_sum=("total_price","sum"),
    mean_total=("total_price","mean"),
    max_order=("total_price","max"),
    unique_categories=("category","nunique")
).reset_index()
final["VIP"]=final["total_sum"]>1000
print(final)

  user_name  total_orders  total_sum  mean_total  max_order  \
0     Alice             1         25        25.0         25   
1      John             2       1700       850.0       1200   

   unique_categories    VIP  
0                  1  False  
1                  2   True  
